# Screen Images Classification Baselines
In this notebook, we run kNN and Random Forest on the screen images as baselines.

In [ ]:
import sys
import os

import torch
from torchvision.transforms import v2
from data.screen_images import CustomEnricoDataset, get_allowed_classes, make_screen_base_transform
from baseline import extract_features, train_evaluate_knn, train_evaluate_random_forest

print("Libraries imported successfully!")


## Data Loading and Preprocessing
We use a smaller image size for classical ML algorithms to keep the feature dimension manageable.

In [ ]:
# Data configuration
BATCH_SIZE = 64
ENRICO_CLASSES = len(get_allowed_classes())

# We resize the image for basic machine learning models so that the flattened feature vector 
# isnt overly huge (e.g., 60x40 means 60*40*3 = 7200 features)
# We use make_screen_base_transform proportional to 300x200
screen_preprocess = make_screen_base_transform(resize=(60, 40))

# Use the data path that you have configured in your previous notebooks
root_dir = "/kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes"

try:
    print(f"Loading data from: {root_dir}")
    train_dataset, val_dataset, test_dataset = CustomEnricoDataset.create_splits(
        root=root_dir,
        val_size=0.1,
        test_size=0.1,
        use_wireframes=False,
        train_transform=screen_preprocess,
        eval_transform=screen_preprocess
    )

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print("Data splits and loaders created successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")


## Feature Extraction
Flatten the multidimensional tensors into 1D vectors per image.

In [ ]:
# Extract features (flatten images into 1D vectors for scikit-learn models)
print("Extracting features for training data...")
X_train, y_train = extract_features(train_loader, flatten=True)

print("Extracting features for testing data...")
X_test, y_test = extract_features(test_loader, flatten=True)

print(f"Train data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")


## k-Nearest Neighbors (kNN) Classifier

In [ ]:
# kNN Baseline
# Setting n_jobs=-1 to use all available CPU cores for faster computation
knn_model, knn_preds = train_evaluate_knn(X_train, y_train, X_test, y_test, n_neighbors=5, n_jobs=-1)


## Random Forest Classifier

In [ ]:
# Random Forest Baseline
# Using 100 estimators as a solid baseline
rf_model, rf_preds = train_evaluate_random_forest(X_train, y_train, X_test, y_test, n_estimators=100, n_jobs=-1, random_state=42)
